# Notebook 00 — Planejamento e Contexto Analítico

**Projeto Mensal 3 — Tratamento de Dados**  
**Base:** Cargos Vagos e Vacâncias do Poder Executivo Federal Civil — competência 03/2026  
**Fonte:** Portal Brasileiro de Dados Abertos / SEGES — Ministério da Gestão e da Inovação em Serviços Públicos  
**URL:** https://dados.gov.br/dados/conjuntos-dados/gestao-de-pessoas-executivo-federal---cargos-vagos-e-vacancias  
**Data de acesso:** 2026

---

## Objetivo deste notebook

Este notebook abre o projeto, antes de qualquer tratamento técnico. Ele estabelece **por que** vamos tratar essa base e **para que** ela serve, respondendo às exigências das seções **6.1 (Planejamento)** e **6.2 (Relação com BI, Big Data Analytics e Data Mining)** do enunciado.

Os notebooks seguintes (`01` a `04`) executam o trabalho técnico de preparação. Este notebook ancora todas as decisões nessas finalidades analíticas.

## Roteiro

| Seção | Etapa do enunciado | Conteúdo |
|---|---|---|
| 1 | 6.1 | Tema, fonte, problema, atores, perguntas analíticas |
| 2 | 6.2 | Aplicação concreta em BI, Big Data Analytics e Data Mining |
| 3 | — | Síntese e ponte para os demais notebooks |


## Setup — carga rápida da base apenas para evidência factual

Este notebook é predominantemente conceitual, mas alguns números (quantidade de órgãos, cargos, distribuição de níveis) são extraídos diretamente da base bruta para ancorar os argumentos em fatos verificáveis.


In [1]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path('..').resolve()
DADOS_BRUTOS = BASE_DIR / 'dados_brutos'

df = pd.read_csv(DADOS_BRUTOS / 'CargosVagosVacancias_202603.csv')

print(f"Base bruta carregada para evidência factual:")
print(f"  Linhas:   {df.shape[0]:,}".replace(',', '.'))
print(f"  Colunas:  {df.shape[1]}")
print(f"  Órgãos:   {df['ORGAO'].nunique()}")
print(f"  Cargos:   {df['CARGO'].nunique()}")
print(f"  Planos:   {df['PLANO_CARREIRA'].nunique()}")

Base bruta carregada para evidência factual:
  Linhas:   12.769
  Colunas:  20
  Órgãos:   211
  Cargos:   2189
  Planos:   154


## 1. Planejamento do tratamento dos dados (etapa 6.1)

### 1.1 Qual é o tema da base?

O tema é a **gestão quantitativa de pessoas no Poder Executivo Federal Civil brasileiro**. A base apresenta, mês a mês, a fotografia do quadro de pessoal de cada um dos órgãos da administração federal direta, autárquica e fundacional — detalhando:

- Quantos cargos foram **aprovados em lei** (teto legal de servidores que o órgão pode ter).
- Quantos foram **distribuídos** ao órgão pelo Órgão Central de Gestão de Pessoas.
- Quantos estão **ocupados** efetivamente.
- Quantos estão **vagos** (à espera de nomeação).
- E, no fluxo do mês, quantos servidores saíram do cargo, separados por sete causas distintas: **aposentadoria, exoneração, demissão, promoção, readaptação, posse em outro cargo inacumulável e falecimento**.

A unidade de análise é a combinação **(órgão × cargo)** — cada linha corresponde a uma carreira específica dentro de um órgão específico, no mês 03/2026.

### 1.2 De onde os dados foram retirados?

| Item | Detalhe |
|---|---|
| **Fonte primária** | Portal Brasileiro de Dados Abertos — `dados.gov.br` |
| **Conjunto de dados** | "Cargos Vagos e Vacâncias do Poder Executivo Federal Civil" |
| **Órgão produtor** | Secretaria de Gestão e Inovação (SEGES) — Ministério da Gestão e da Inovação em Serviços Públicos (MGI) |
| **Arquivo utilizado** | `CargosVagosVacancias_202603.ods`, aba `por_Órgao_e_Cargo` |
| **Competência** | Março/2026 (snapshot mensal) |
| **Periodicidade da publicação** | Mensal — séries históricas disponíveis no mesmo portal |
| **Licença** | Dados abertos sob CC BY 4.0 |
| **Dicionário oficial** | Disponível em PDF no mesmo conjunto de dados |
| **Data de acesso** | 2026 |

A escolha pela aba `por_Órgao_e_Cargo` (12.769 linhas) — em vez da aba `por_Órgao` (211 linhas agregadas) — se justifica porque a base detalhada permite todas as análises da base agregada e mais várias outras, enquanto o contrário não é possível.

### 1.3 Qual problema ou situação essa base ajuda a compreender?

A base permite enxergar fenômenos que estão no centro da política de gestão de pessoas no setor público federal:

1. **Déficit estrutural de servidores.** Quando `qtd_aprovada` (teto legal) é muito maior que `qtd_ocupada`, o órgão opera abaixo da capacidade autorizada por lei. Isso reflete em fila de atendimento (INSS), morosidade fiscal (Receita Federal), gargalos universitários, etc.

2. **Pressão de reposição via aposentadoria.** O Executivo Federal está envelhecendo: aposentadoria é, de longe, a principal causa de vacância (≈75% do total em 03/2026). Saber **onde** essa pressão é maior orienta calendários de concurso público.

3. **Estoque de cargos em extinção.** Cerca de 28% dos cargos da base estão marcados como `em_extinção` — vestígios de carreiras antigas que ainda permanecem na estrutura legal, mesmo com poucos servidores ativos. Mapear essas carreiras é parte da modernização da administração.

4. **Concentração desigual de pessoal.** A distribuição é altamente assimétrica — alguns poucos cargos institucionais (Técnico Administrativo em Educação, Polícia Federal, Auditor Fiscal) concentram milhares de servidores, enquanto centenas de cargos têm 1 ou 2 ocupantes. Isso afeta decisões sobre fusão, extinção e racionalização de carreiras.

5. **Reorganização ministerial recorrente.** A mesma atribuição governamental aparece sob nomes de órgãos diferentes ao longo do tempo (ex.: extinções e recriações de ministérios). A base preserva esses registros históricos.

### 1.4 Quem poderia usar esses dados para tomar decisão?

| Ator | Decisão que toma com base nesses dados |
|---|---|
| **MGI / SEGES** | Calendário de concursos públicos; redistribuição de vagas entre órgãos; políticas de carreira |
| **Ministério do Planejamento e Orçamento** | Previsão da folha de pagamento; impacto orçamentário de reposições |
| **TCU** (controle externo) | Fiscalização do cumprimento dos tetos legais por órgão |
| **CGU** (controle interno) | Auditorias sobre cessões e movimentações de pessoal |
| **Comissões do Congresso** (Administração Pública, Trabalho, Orçamento) | Pareceres sobre projetos de lei de reestruturação de carreiras |
| **Dirigentes dos próprios órgãos** | Solicitação de novos concursos; planejamento de sucessão |
| **Sindicatos de servidores** | Acompanhamento da força de trabalho da categoria |
| **Pesquisadores e jornalismo de dados** | Estudos sobre o Estado brasileiro, reportagens de gestão pública |

### 1.5 Perguntas analíticas que essa base permite responder

Essas são as perguntas que orientam todas as decisões técnicas dos próximos notebooks. Cada coluna mantida, cada feature criada, cada transformação aplicada tem que servir a pelo menos uma delas:

**Perguntas descritivas (BI clássico):**
1. Quais órgãos do Executivo Federal têm mais vagas em aberto em 03/2026?
2. Qual a taxa de ocupação média do governo federal? E por nível de escolaridade?
3. Quais são os 10 cargos com maior quantitativo total no Executivo?
4. Quanto pesa a aposentadoria como causa de vacância, comparada às outras seis causas?
5. Quantos cargos estão formalmente em extinção e em quais órgãos eles ainda têm ocupantes?

**Perguntas comparativas / segmentadas:**
6. Cargos de Nível Superior têm taxa de ocupação maior ou menor que cargos de Nível Intermediário?
7. Órgãos pequenos (poucas vagas aprovadas) têm comportamento de vacância diferente dos grandes?
8. A vacância por exoneração se concentra em algum tipo específico de carreira?

**Perguntas estruturais / de descoberta (Data Mining):**
9. É possível agrupar os órgãos em "perfis" típicos (ex.: órgão fiscalizador, órgão prestador de serviço, órgão regulador) a partir do perfil quantitativo do seu quadro?
10. Existem cargos que tendem a coexistir nos mesmos órgãos (regras de associação)?
11. Há outliers que mereçam tratamento separado (cargos institucionais gigantes)?


### Evidência factual das afirmações acima

As principais afirmações desta seção podem ser verificadas diretamente na base bruta:


In [2]:
# Confirmar a estrutura da base
print("=== ESTRUTURA DA BASE BRUTA ===")
print(f"Linhas:  {df.shape[0]:,}".replace(',', '.'))
print(f"Colunas: {df.shape[1]}")
print(f"Órgãos distintos:           {df['ORGAO'].nunique()}")
print(f"Cargos (códigos) distintos: {df['CARGO'].nunique()}")
print(f"Planos de carreira:         {df['PLANO_CARREIRA'].nunique()}")

# Quantidade total de servidores ativos no Executivo Federal Civil em 03/2026
print(f"\nServidores ATIVOS no Executivo Federal Civil em 03/2026: {df['OCUPADA'].sum():,}".replace(',', '.'))
print(f"Vagas APROVADAS em lei (teto legal):                     {df['APROVADA'].sum():,}".replace(',', '.'))
print(f"Vagas em ABERTO (não preenchidas):                       {df['VAGA'].sum():,}".replace(',', '.'))

taxa_ocup = df['OCUPADA'].sum() / df['APROVADA'].sum() * 100
print(f"Taxa de ocupação federal geral:                          {taxa_ocup:.1f}%")

=== ESTRUTURA DA BASE BRUTA ===
Linhas:  12.769
Colunas: 20
Órgãos distintos:           211
Cargos (códigos) distintos: 2189
Planos de carreira:         154

Servidores ATIVOS no Executivo Federal Civil em 03/2026: 468.264
Vagas APROVADAS em lei (teto legal):                     700.413
Vagas em ABERTO (não preenchidas):                       232.149
Taxa de ocupação federal geral:                          66.9%


In [3]:
# Confirmar que aposentadoria é a principal causa de vacância
cols_vac = [c for c in df.columns if c.startswith('VACANCIA_')]
totais_vac = df[cols_vac].sum().sort_values(ascending=False)
total_vacancias = totais_vac.sum()

print("=== VACÂNCIAS POR TIPO — março/2026 ===\n")
for tipo, qtd in totais_vac.items():
    pct = qtd / total_vacancias * 100
    nome_curto = tipo.replace('VACANCIA_POR_', '').replace('_', ' ').title()
    print(f"  {nome_curto:<30} {qtd:>8,}  ({pct:>5.1f}%)".replace(',', '.'))

print(f"\n  {'TOTAL':<30} {total_vacancias:>8,}".replace(',', '.'))

=== VACÂNCIAS POR TIPO — março/2026 ===

  Aposentadoria                   214.829  ( 76.7%)
  Posse Cargo Inac                 22.983  (  8.2%)
  Exoneracao                       20.677  (  7.4%)
  Falecimento                      16.943  (  6.0%)
  Demissao                          4.084  (  1.5%)
  Promocao                            530  (  0.2%)
  Readaptacao                          79  (  0.0%)

  TOTAL                           280.125


In [4]:
# Confirmar a presença de cargos em extinção
em_ext = df['CARGO_EM_EXTINCAO'].value_counts()
pct_ext = em_ext.get('S', 0) / len(df) * 100
print(f"Cargos em extinção: {em_ext.get('S', 0):,} de {len(df):,} ({pct_ext:.1f}%)".replace(',', '.'))

# Quais órgãos concentram mais cargos em extinção?
print(f"\nTop 5 órgãos com mais cargos em extinção:")
ext_por_orgao = df[df['CARGO_EM_EXTINCAO'] == 'S'].groupby('SIGLA_ORGAO').size().nlargest(5)
print(ext_por_orgao.to_string())

Cargos em extinção: 3.577 de 12.769 (28.0%)

Top 5 órgãos com mais cargos em extinção:
SIGLA_ORGAO
MS           159
MGI          130
EX-TER/AP     96
AGU           75
EX-TER/RR     75


## 2. Relação da base com Big Data Analytics, Data Mining e Business Intelligence (etapa 6.2)

> O enunciado adverte: *"Não basta copiar definições genéricas. Você deverá aplicar os conceitos ao seu próprio tema."*  
> Esta seção segue essa orientação: cada conceito é apresentado com aplicação concreta à base de Cargos Vagos e Vacâncias.

### 2.1 Aplicação em Business Intelligence

Business Intelligence (BI) é o conjunto de práticas de **coleta, organização, análise e apresentação** de informações operacionais para apoiar decisões gerenciais. No caso desta base, BI se manifesta da seguinte forma:

**a) Dashboards de monitoramento operacional.** A combinação `(órgão, cargo)` é uma chave dimensional natural. As colunas quantitativas (`aprovada`, `distribuida`, `ocupada`, `vaga`) são medidas aditivas. Isso é exatamente o desenho de um cubo OLAP típico — pode ser carregado em Power BI, Looker Studio, Tableau ou Metabase para gerar:

- **Painel executivo do MGI:** taxa de ocupação federal, vagas em aberto, ranking de órgãos deficitários, evolução mensal das vacâncias por tipo.
- **Painel por ministério:** o gestor de um ministério vê apenas seus órgãos vinculados, seus cargos, sua pirâmide de vacâncias.
- **Painel de carreiras:** distribuição de uma carreira específica (ex.: "Auditor Fiscal") entre os órgãos onde existe.

**b) Indicadores (KPIs) de gestão de pessoas.** A partir das colunas brutas é possível derivar indicadores acionáveis, que serão criados no Notebook 03 como features:

- **Taxa de ocupação** = `ocupada / aprovada` — mede o quão preenchido está o quadro legal.
- **Taxa de vacância no mês** = `(total_vacancias_mes) / ocupada` — velocidade de saída de servidores.
- **Déficit nominal** = `aprovada - ocupada` — quantidade absoluta de servidores que faltam.
- **Razão aposentadoria / outras vacâncias** — proxy de envelhecimento da carreira.

**c) Drill-down hierárquico.** A estrutura permite navegação top-down típica de BI:
`Executivo Federal → Ministério → Órgão → Plano de Carreira → Cargo Específico`

**d) Self-service e governança de dados.** Uma vez tratada, a base pode ser publicada em um catálogo de dados (como o que entregaremos no Notebook 04), permitindo que diferentes áreas a consumam sem reprocessar o ETL.

### 2.2 Aplicação em Big Data Analytics

Big Data Analytics (BDA) lida com dados que apresentam pelo menos uma das clássicas características dos "5 Vs": **Volume, Variedade, Velocidade, Veracidade e Valor**. A nossa base, vista de forma isolada, **não é volumosa** (12.769 linhas cabem em qualquer computador). Mas a análise honesta sobre BDA é a seguinte:

**a) Volume — não, mas com ressalvas.** O snapshot mensal é pequeno. **No entanto**, a série histórica acumulada é grande: o SEGES publica a base mensalmente há mais de uma década. Encadear ≥120 meses gera mais de 1,5 milhão de registros — começa a justificar arquitetura analítica mais robusta (pipelines automatizados, particionamento por competência).

**b) Variedade — sim.** A base mistura tipos heterogêneos: identificadores hierárquicos (`cod_orgao`, `cod_cargo`), categorias (`nivel`, `plano_carreira`, `em_extincao`), quantidades inteiras, e potencialmente data (`ano_mes`). Em projetos de BDA reais, esta base seria integrada a outras fontes:

- **SIAPE** (folha de pagamento) para cruzar com perfil etário e remuneração.
- **RAIS / CAGED** para comparar com mercado de trabalho.
- **DADOS** abertos sobre concursos públicos em curso.
- **Estatísticas IBGE** sobre população economicamente ativa.

A integração dessas fontes traz volume + variedade reais.

**c) Velocidade — moderada.** Publicação mensal, processamento batch. Não é stream em tempo real — mas para o caso de uso (planejamento estratégico de pessoal), latência mensal é suficiente.

**d) Veracidade — alta.** Fonte governamental oficial, com dicionário publicado e regras de extração documentadas. Os problemas de qualidade que identificamos no Notebook 01 (nulos em NIVEL, inconsistências entre OCUPADA + VAGA vs. DISTRIBUIDA) são marginais e justificáveis.

**e) Valor — alto.** Decisões de concurso público envolvem bilhões de reais por ano em folha; um modelo analítico baseado nesta base impacta diretamente esse gasto.

**Em resumo:** isoladamente esta base **não é Big Data**, mas é a **unidade básica de um pipeline de BDA** quando combinada com séries históricas e com outras bases federais.

### 2.3 Aplicação em Data Mining

Data Mining (DM) é o processo de **descoberta de padrões, relações e estruturas previamente desconhecidas** em grandes massas de dados. Distingue-se de BI porque o BI **responde a perguntas pré-formuladas**, enquanto o DM **gera perguntas novas a partir dos dados**.

Para a base de Cargos Vagos e Vacâncias, as tarefas clássicas de DM se aplicam assim:

**a) Clustering (agrupamento).** A pergunta natural é: *"Os 210 órgãos do Executivo Federal podem ser agrupados em perfis distintos com base na composição do seu quadro?"*

Variáveis úteis: taxa de ocupação, taxa de aposentadoria, % de cargos em extinção, % de cargos de nível superior, tamanho médio do quadro. Algoritmos: **K-Means**, **Hierarchical Clustering**, **DBSCAN**. Resultado esperado: clusters como "órgãos prestadores de serviço com alta rotatividade", "órgãos reguladores enxutos e estáveis", "órgãos universitários com grandes quadros docentes", etc.

**b) Detecção de anomalias (outliers).** *Quais cargos têm comportamento atípico em relação à norma da base?* Já implementamos uma versão simples disso no Notebook 02 (flag `is_outlier` por critério IQR em `qtd_aprovada`). Em DM mais sofisticado, técnicas como **Isolation Forest** ou **Local Outlier Factor** poderiam identificar:

- Cargos com taxa de vacância anormalmente alta sem justificativa estrutural.
- Órgãos onde a soma das vacâncias está muito acima do esperado para o tamanho do quadro.
- Inconsistências entre cargos similares em órgãos similares.

**c) Regras de associação.** *Quais cargos tendem a coexistir nos mesmos órgãos?* Algoritmos como **Apriori** ou **FP-Growth** poderiam identificar:

- "Todo órgão que tem cargo X também tem cargo Y" (estrutura típica de órgão de fiscalização).
- "Cargos de apoio administrativo aparecem com cargos finalísticos específicos".

**d) Classificação supervisionada.** Embora a base atual não tenha rótulo natural óbvio, ela suporta tarefas como:

- *Dado o perfil de um cargo, classificar se ele tende a estar "em extinção" ou "ativo".* Modelos: **Árvores de Decisão**, **Random Forest**, **Logistic Regression**. Variáveis preditoras: nível, plano de carreira, tamanho do quadro, perfil de vacância.

**e) Análise preditiva (regressão).** Combinando esta base com a série histórica:

- *Prever quantas vacâncias por aposentadoria cada órgão terá nos próximos 12 meses* — informação direta para planejamento de concursos. Modelos: **séries temporais (ARIMA/Prophet)**, **regressão multivariada**, **gradient boosting**.

### 2.4 Resumo: o que a base permite por área

| Área | Tarefa concreta | Variáveis-chave usadas | Entrega típica |
|---|---|---|---|
| **BI** | Dashboard executivo de gestão de pessoal | `nome_orgao`, `qtd_aprovada`, `qtd_ocupada`, `qtd_vaga` | Painel Power BI / Looker |
| **BI** | Indicadores de RH público (taxa de ocupação, déficit) | `qtd_*`, `vac_*` | KPIs + rankings |
| **BDA** | Pipeline mensal de ingestão com versionamento histórico | Toda a base + metadado de competência | Data Lake + catálogo |
| **BDA** | Integração com SIAPE/RAIS para análise cruzada | `cod_orgao`, `cod_cargo` como chaves | Modelo dimensional integrado |
| **DM** | Clustering de órgãos por perfil de quadro | Variáveis derivadas (taxas, proporções) | Segmentação para tomada de decisão |
| **DM** | Detecção de cargos com vacância anômala | `vac_*` / `qtd_ocupada` | Lista priorizada para auditoria |
| **DM** | Regras de associação entre cargos | `cod_cargo` × `cod_orgao` | Padrões estruturais documentados |


## 3. Síntese e ponte para os próximos notebooks

Este notebook estabeleceu o **porquê** do projeto. As decisões técnicas dos próximos quatro notebooks serão guiadas por ele:

| Notebook | Decisão técnica | Justificativa ancorada neste notebook |
|---|---|---|
| **01** — Diagnóstico | Mapear nulos, tipos, inconsistências | Sem qualidade não há BI confiável (seção 2.1) |
| **02** — Limpeza | Preservar outliers como flag, não removê-los | Os "outliers" são justamente os cargos institucionais mais relevantes para análise (seção 1.5, p.11) |
| **02b** — AED | Histograma com escala log, boxplot por nível | Reflete a alta assimetria intrínseca da base (seção 1.3, p.4) |
| **03** — Feature Engineering | Criar `taxa_ocupacao`, `total_vacancias`, faixas de tamanho | São exatamente os KPIs descritos em 2.1.b e variáveis para clustering em 2.3.a |
| **04** — Dataset final + catálogo | Documentar cada coluna com sua origem e uso | Permite que os atores listados em 1.4 reusem a base sem reprocessar |

**Próximo passo:** abrir o `01_diagnostico_qualidade.ipynb` para iniciar o trabalho técnico.

---

## Referências usadas neste notebook

- BRASIL. **Lei nº 8.112/90**, art. 33 — Vacância no Regime Jurídico Único.
- BRASIL. **SEGES/MGI.** Conjunto de dados "Cargos Vagos e Vacâncias do Poder Executivo Federal Civil". Disponível em: `dados.gov.br`.
- BRASIL. **SEGES/MGI.** Dicionário oficial dos campos da base (PDF anexo ao conjunto de dados).
- CASTRO, L. N.; FERRARI, D. G. *Introdução à mineração de dados.* Saraiva, 2016 — Cap. 1 (taxonomia das tarefas de DM).
- GOLDSCHMIDT, R.; BEZERRA, E.; PASSOS, E. *Data mining: conceitos, técnicas, algoritmos.* Elsevier, 2015.
- CERCHIARI DE ANDRADE, A. L. *Preparação e análise exploratória de dados.* SAGAH.
